# 006 – Geometrie-Puzzles

Sechs Knobelaufgaben rund um Geometrie, Gitter und Abzählen.

Der rote Faden: **Jedes Puzzle wird zweimal gelöst** – einmal durch Nachdenken (Formel)
und einmal durch stures Ausprobieren (Programm). Wenn beide Wege dasselbe Ergebnis
liefern, stimmt es vermutlich. Und wenn nicht, hat man etwas gelernt.

Dieses Notebook ist eigenständig, es braucht nichts aus den vorherigen Notebooks.

## Puzzle 1 – Wie viele Quadrate stecken in einem Schachbrett?

Ein Schachbrett hat 8 × 8 Felder. Gesucht sind **alle** Quadrate, die man auf den Linien
des Bretts abzeichnen kann: die 64 kleinen 1×1-Quadrate, die 2×2-Quadrate, ... bis zum
ganzen Brett.

Die meisten tippen spontan auf 64 – es sind deutlich mehr.

Zugabe: Wie viele **Rechtecke** sind es? Tipp: Ein Rechteck ist eindeutig festgelegt,
sobald man zwei der 9 senkrechten und zwei der 9 waagerechten Linien auswählt.

In [1]:
def quadrate_ausprobieren(n):
    # alle Seitenlaengen und alle moeglichen Positionen durchgehen
    anzahl = 0
    for k in range(1, n + 1):
        for x in range(n - k + 1):
            for y in range(n - k + 1):
                anzahl += 1
    return anzahl

def quadrate_formel(n):
    # 1^2 + 2^2 + ... + n^2
    return n * (n + 1) * (2 * n + 1) // 6

def rechtecke_formel(n):
    linien = n + 1
    paare = linien * (linien - 1) // 2      # 2 aus (n+1) Linien auswaehlen
    return paare * paare

print("Ausprobiert:", quadrate_ausprobieren(8))
print("Formel:     ", quadrate_formel(8))
print("Rechtecke:  ", rechtecke_formel(8))
print()
print(f"{'Brett':>7} {'Quadrate':>9} {'Rechtecke':>10}")
for n in range(1, 9):
    print(f"{str(n) + 'x' + str(n):>7} {quadrate_formel(n):>9} {rechtecke_formel(n):>10}")

Ausprobiert: 204
Formel:      204
Rechtecke:   1296

  Brett  Quadrate  Rechtecke
    1x1         1          1
    2x2         5          9
    3x3        14         36
    4x4        30        100
    5x5        55        225
    6x6        91        441
    7x7       140        784
    8x8       204       1296


## Puzzle 2 – Der Satz von Pick

Ein Vieleck, dessen Ecken alle auf Gitterpunkten liegen, hat die Fläche

`A = I + R/2 − 1`

wobei I = Anzahl der Gitterpunkte **innerhalb** und R = Anzahl der Gitterpunkte **auf dem Rand**.

Das ist ziemlich verblüffend: Man muss überhaupt nichts messen, nur Punkte zählen.

Wir zählen die Punkte per Programm und vergleichen mit der Schnürsenkelformel.
Kleiner Trick für den Rand: Auf einer Kante von (x₁|y₁) nach (x₂|y₂) liegen genau
`ggT(|Δx|, |Δy|)` Gitterpunkte (den Startpunkt mitgezählt).

In [2]:
from math import gcd

def polygon_flaeche(punkte):
    n = len(punkte)
    summe = 0
    for i in range(n):
        x1, y1 = punkte[i]
        x2, y2 = punkte[(i + 1) % n]
        summe += x1 * y2 - x2 * y1
    return abs(summe) / 2

def randpunkte(punkte):
    n = len(punkte)
    anzahl = 0
    for i in range(n):
        x1, y1 = punkte[i]
        x2, y2 = punkte[(i + 1) % n]
        anzahl += gcd(abs(x2 - x1), abs(y2 - y1))
    return anzahl

def auf_rand(P, punkte):
    n = len(punkte)
    for i in range(n):
        A = punkte[i]
        B = punkte[(i + 1) % n]
        kreuz = (B[0] - A[0]) * (P[1] - A[1]) - (B[1] - A[1]) * (P[0] - A[0])
        if kreuz == 0 and min(A[0], B[0]) <= P[0] <= max(A[0], B[0]) \
                and min(A[1], B[1]) <= P[1] <= max(A[1], B[1]):
            return True
    return False

def echt_innen(P, punkte):
    if auf_rand(P, punkte):
        return False
    x, y = P
    drin = False
    n = len(punkte)
    for i in range(n):
        x1, y1 = punkte[i]
        x2, y2 = punkte[(i + 1) % n]
        if (y1 > y) != (y2 > y):
            if x < x1 + (y - y1) * (x2 - x1) / (y2 - y1):
                drin = not drin
    return drin

def innere_punkte(punkte):
    xs = [p[0] for p in punkte]
    ys = [p[1] for p in punkte]
    anzahl = 0
    for x in range(min(xs), max(xs) + 1):
        for y in range(min(ys), max(ys) + 1):
            if echt_innen((x, y), punkte):
                anzahl += 1
    return anzahl

figur = [(0, 0), (6, 0), (6, 4), (3, 7), (0, 4)]

I = innere_punkte(figur)
R = randpunkte(figur)
print(f"innere Punkte  I = {I}")
print(f"Randpunkte     R = {R}")
print(f"Satz von Pick:   A = {I} + {R}/2 - 1 = {I + R / 2 - 1}")
print(f"Schnuersenkel:   A = {polygon_flaeche(figur)}")

innere Punkte  I = 24
Randpunkte     R = 20
Satz von Pick:   A = 24 + 20/2 - 1 = 33.0
Schnuersenkel:   A = 33.0


## Puzzle 3 – Das rechtwinklige Dreieck mit Umfang 1000

Gesucht ist ein rechtwinkliges Dreieck mit **ganzzahligen** Seiten a < b < c und
a + b + c = 1000. Gibt es das überhaupt? Und wenn ja, wie viele?

Zweiter Teil: Welcher Umfang bis 1000 lässt die **meisten** verschiedenen rechtwinkligen
Dreiecke zu? Statt für jeden Umfang neu zu suchen, gehen wir alle Paare (a, b) einmal
durch und sortieren die Treffer nach ihrem Umfang – viel schneller.

In [3]:
import math
from collections import Counter

def tripel_mit_umfang(u):
    treffer = []
    for a in range(1, u // 3):
        for b in range(a + 1, (u - a) // 2 + 1):
            c = u - a - b
            if a * a + b * b == c * c:
                treffer.append((a, b, c))
    return treffer

for u in [12, 24, 30, 1000]:
    print(f"Umfang {u:>4}: {tripel_mit_umfang(u)}")

zaehler = Counter()
for a in range(1, 500):
    for b in range(a + 1, 500):
        c_quadrat = a * a + b * b
        c = math.isqrt(c_quadrat)
        if c * c == c_quadrat and a + b + c <= 1000:
            zaehler[a + b + c] += 1

bester = max(zaehler, key=lambda u: zaehler[u])
print()
print(f"Meiste Loesungen hat der Umfang {bester}: {zaehler[bester]} Dreiecke")
for seiten in tripel_mit_umfang(bester):
    print("   ", seiten)

Umfang   12: [(3, 4, 5)]
Umfang   24: [(6, 8, 10)]
Umfang   30: [(5, 12, 13)]
Umfang 1000: [(200, 375, 425)]

Meiste Loesungen hat der Umfang 840: 8 Dreiecke
    (40, 399, 401)
    (56, 390, 394)
    (105, 360, 375)
    (120, 350, 370)
    (140, 336, 364)
    (168, 315, 357)
    (210, 280, 350)
    (240, 252, 348)


## Puzzle 4 – Wie viele Dreiecke kann man legen?

Du hast Stäbe der Längen 1, 2, 3, ..., 10 – jede Länge genau einmal.
Wie viele verschiedene Dreiecke lassen sich daraus legen?

Bedingung ist die Dreiecksungleichung: Bei a ≤ b ≤ c muss `a + b > c` gelten.
Aus 10 Stäben kann man 120 Dreier-Kombinationen bilden – aber längst nicht jede
ergibt ein Dreieck.

In [4]:
from itertools import combinations

def anzahl_dreiecke(laengen):
    anzahl = 0
    for a, b, c in combinations(sorted(laengen), 3):
        if a + b > c:
            anzahl += 1
    return anzahl

staebe = list(range(1, 11))
print(f"Kombinationen insgesamt: {len(list(combinations(staebe, 3)))}")
print(f"davon echte Dreiecke:    {anzahl_dreiecke(staebe)}")

print()
print("Wie waechst das mit der Anzahl der Staebe?")
for n in range(3, 13):
    moeglich = len(list(combinations(range(1, n + 1), 3)))
    echt = anzahl_dreiecke(range(1, n + 1))
    print(f"Staebe 1..{n:>2}: {echt:>4} von {moeglich:>4} Kombinationen ({100 * echt / moeglich:.1f} %)")

Kombinationen insgesamt: 120
davon echte Dreiecke:    50

Wie waechst das mit der Anzahl der Staebe?
Staebe 1.. 3:    0 von    1 Kombinationen (0.0 %)
Staebe 1.. 4:    1 von    4 Kombinationen (25.0 %)
Staebe 1.. 5:    3 von   10 Kombinationen (30.0 %)
Staebe 1.. 6:    7 von   20 Kombinationen (35.0 %)
Staebe 1.. 7:   13 von   35 Kombinationen (37.1 %)
Staebe 1.. 8:   22 von   56 Kombinationen (39.3 %)
Staebe 1.. 9:   34 von   84 Kombinationen (40.5 %)
Staebe 1..10:   50 von  120 Kombinationen (41.7 %)
Staebe 1..11:   70 von  165 Kombinationen (42.4 %)
Staebe 1..12:   95 von  220 Kombinationen (43.2 %)


## Puzzle 5 – Das verstümmelte Schachbrett

Von einem 8 × 8-Schachbrett werden zwei **gegenüberliegende Eckfelder** entfernt.
Bleiben 62 Felder. Kann man die lückenlos mit 31 Dominosteinen (je 2 × 1 Felder) belegen?

Man kann sich lange totprobieren – oder einen Moment auf die Farben schauen:
Jeder Dominostein bedeckt immer **ein weißes und ein schwarzes** Feld. Die beiden
entfernten Ecken haben aber dieselbe Farbe. Also stimmt die Bilanz nicht mehr.

Danach lassen wir einen Computer alle Belegungen zählen und schauen, ob er dem
Farbargument zustimmt.

In [5]:
from functools import lru_cache

def farbzaehlung(n, entfernt=()):
    weiss = schwarz = 0
    for x in range(n):
        for y in range(n):
            if (x, y) in entfernt:
                continue
            if (x + y) % 2 == 0:
                weiss += 1
            else:
                schwarz += 1
    return weiss, schwarz

def anzahl_belegungen(breite, hoehe, entfernt=()):
    entfernt = set(entfernt)
    frei = frozenset((x, y) for x in range(breite) for y in range(hoehe)
                     if (x, y) not in entfernt)

    @lru_cache(maxsize=None)
    def zaehle(rest):
        if not rest:
            return 1                       # alles belegt: eine gueltige Loesung
        x, y = min(rest)                   # immer das erste freie Feld zuerst belegen
        summe = 0
        for nachbar in [(x + 1, y), (x, y + 1)]:
            if nachbar in rest:
                summe += zaehle(rest - {(x, y), nachbar})
        return summe

    return zaehle(frei)

ecken = ((0, 0), (7, 7))                   # zwei gegenueberliegende Ecken, gleiche Farbe
w, s = farbzaehlung(8, ecken)
print(f"nach dem Entfernen: {w} weisse und {s} schwarze Felder")
print("Ein Domino bedeckt immer 1 weisses + 1 schwarzes Feld ->",
      "moeglich" if w == s else "unmoeglich")
print()

print("volles Brett  8x8:              ", anzahl_belegungen(8, 8), "Belegungen")
print("ohne zwei gleichfarbige Ecken:  ", anzahl_belegungen(8, 8, ecken), "Belegungen")
print("ohne zwei verschiedenfarbige:   ",
      anzahl_belegungen(8, 8, ((0, 0), (0, 1))), "Belegungen")
print()
for n in [2, 4, 6, 8]:
    print(f"{n}x{n}-Brett: {anzahl_belegungen(n, n)} Belegungen")

nach dem Entfernen: 30 weisse und 32 schwarze Felder
Ein Domino bedeckt immer 1 weisses + 1 schwarzes Feld -> unmoeglich

volles Brett  8x8:               12988816 Belegungen
ohne zwei gleichfarbige Ecken:   0 Belegungen
ohne zwei verschiedenfarbige:    6494408 Belegungen

2x2-Brett: 2 Belegungen
4x4-Brett: 36 Belegungen
6x6-Brett: 6728 Belegungen
8x8-Brett: 12988816 Belegungen


## Puzzle 6 – Wege im Gitter

In einem Straßengitter aus 6 × 4 Blöcken willst du von der linken unteren Ecke zur
rechten oberen. Erlaubt sind nur Schritte nach **rechts** und nach **oben**.
Wie viele kürzeste Wege gibt es?

Zwei Denkweisen, dasselbe Ergebnis:

* **Kombinatorisch:** Jeder Weg besteht aus 6 Rechts- und 4 Hoch-Schritten, also aus
  10 Schritten, von denen man 4 als „hoch" auswählt: `C(10, 4)`.
* **Rekursiv:** In jede Kreuzung kommt man von links oder von unten – also ist die
  Anzahl der Wege dorthin die Summe der beiden Nachbarn. Das ergibt genau das
  Pascalsche Dreieck, hier als Tabelle aufgebaut.

Und dann die Zusatzfrage: Eine Kreuzung wird gesperrt. Wie viele Wege bleiben übrig?

In [6]:
import math

def wege(breite, hoehe, gesperrt=()):
    gesperrt = set(gesperrt)
    tabelle = [[0] * (breite + 1) for _ in range(hoehe + 1)]
    tabelle[0][0] = 1
    for y in range(hoehe + 1):
        for x in range(breite + 1):
            if (x, y) in gesperrt:
                tabelle[y][x] = 0
                continue
            if x > 0:
                tabelle[y][x] += tabelle[y][x - 1]     # von links
            if y > 0:
                tabelle[y][x] += tabelle[y - 1][x]     # von unten
    return tabelle[hoehe][breite]

print("rekursiv (Tabelle):", wege(6, 4))
print("kombinatorisch:    ", math.comb(10, 4))
print()

gesperrt = [(3, 2)]
print(f"mit gesperrter Kreuzung (3|2): {wege(6, 4, gesperrt)} Wege")

# Kontrolle ohne Tabelle: alle Wege minus die Wege, die durch (3|2) fuehren
durch = math.comb(3 + 2, 2) * math.comb(3 + 2, 2)   # hin bis (3|2), dann weiter zum Ziel
print(f"Kontrolle: Wege durch (3|2) = {durch}, also 210 - {durch} = {210 - durch}")
print()

print("Das Gitter von unten links aus gesehen (Anzahl Wege je Kreuzung):")
for y in range(4, -1, -1):
    zeile = [wege(x, y) for x in range(7)]
    print("  " + " ".join(f"{z:>4}" for z in zeile))

rekursiv (Tabelle): 210
kombinatorisch:     210

mit gesperrter Kreuzung (3|2): 110 Wege
Kontrolle: Wege durch (3|2) = 100, also 210 - 100 = 110

Das Gitter von unten links aus gesehen (Anzahl Wege je Kreuzung):
     1    5   15   35   70  126  210
     1    4   10   20   35   56   84
     1    3    6   10   15   21   28
     1    2    3    4    5    6    7
     1    1    1    1    1    1    1


## Und jetzt du

Ideen zum Weiterknobeln:

1. **Quadrate schräg:** In Puzzle 1 haben wir nur achsenparallele Quadrate gezählt.
   Zählt man auch die schrägen mit (Ecken auf Gitterpunkten), kommt man auf eine
   viel größere Zahl. Wie groß?
2. **Pick rückwärts:** Zeichne ein Vieleck mit genau 10 inneren Punkten und Fläche 12.
   Wie viele Randpunkte muss es haben?
3. **Dominos:** Wie viele Belegungen hat ein 2 × n-Streifen? Lass es für n = 1 bis 10
   ausrechnen – die Zahlenreihe kennst du wahrscheinlich.
4. **Wege:** Wie viele Wege gibt es, wenn man zusätzlich diagonale Schritte erlaubt?
5. **Pythagoras:** Welche Umfänge unter 1000 lassen *gar kein* rechtwinkliges Dreieck zu?
   Fällt bei den Zahlen etwas auf?